In [1]:
# ============================================================
# Batch Training-Condition Generator
# ----------------------------------------------------------
# Generates and pickles every synthetic training condition file consumed
# by the CRHM/CRPM/IRHM notebooks: for each combination of inhibition
# strength (Ki), spectral overlap, low-detectability weighting (Scenario
# 1/2 vs. 3), and random seed, simulates the ground-truth kinetics,
# projects into noisy synthetic Raman spectra, and saves the resulting
# {t_fin, wv_fin, D_full, C0, ST_ground} bundle to
# "Train Conditions/Scenario <n>/TC_<Ki1>_<Ki2>_<seed>_<overlap>_<noise>.pkl".
# ============================================================

import math
import numpy as np
import pandas as pd
from datetime import datetime
import time as time_mod

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from pyomo.environ import *
from pyomo.dae import *

from scipy.integrate import odeint, solve_ivp
from scipy.interpolate import CubicSpline
from sklearn.metrics import r2_score, mean_squared_error

from joblib import Parallel, delayed

import pickle
import random

import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import gc

# pytorch relates imports
import torch
import torch.nn as nn
import torch.optim as optim

# imports from captum library
from captum.attr import IntegratedGradients, Saliency, Occlusion

# Generate Synthetic Raman Data

In [2]:
# Ground-truth data-generating process: two enzyme-catalyzed reactions
# A -> B -> C, each following Michaelis-Menten kinetics with noncompetitive
# product inhibition by C.
def case_study1(t, y, Ki):
    Ca, Cb, Cc = y
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    v1max = 4.5
    v2max = 2.5
    
    Km1 = 5
    Km2 = 5
    
    Ki1 = Ki[0]
    Ki2 = Ki[1]
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    inh1 = (1 + (Cc/Ki1))
    inh2 = (1 + (Cc/Ki2))
    
    # Noncompetitive Inhibition    
    v1 = (v1max*Ca)/((Ca + Km1)*inh1)
    v2 = (v2max*Cb)/((Cb + Km2)*inh2)

    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    
    dCadt = -v1
    dCbdt = v1 - v2
    dCcdt = v2
    
    # Order matches the y = [Ca, Cb, Cc] state vector passed in by solve_ivp.
    return [dCadt, dCbdt, dCcdt]

In [3]:
# Simulates noise-free concentration trajectories for the two fixed initial
# conditions (Ca0 = 0.5, 0.2 mM) used throughout the training set, over a
# shared time grid of length num_time_steps.
def syn_GT_no_noise(num_time_steps, Ki):
    
    ICs = np.array([[0.5, 0, 0],
                    [0.2, 0, 0]])
    
    final_dict = {}
    
    for i in range(len(ICs)):
        
        ic = ICs[i]
        tspan = (0, 20)
        
        teval = np.linspace(tspan[0], tspan[1], num_time_steps)
        
        sol = solve_ivp(case_study1, 
                        tspan, 
                        ic,
                        args = Ki,
                        t_eval = teval,
                        method = "Radau")
        
        key = "Experiment " + str(i+1)
        
        final_dict[key] = [sol.t, sol.y]
    
    return final_dict

In [4]:
# Projects concentration trajectories into synthetic Raman spectra:
# randomly draws one pure spectrum per species from a library of
# biologically-relevant Raman spectra (seeded by RSeed_STchoice), applies
# the low-detectability weighting (id_of_LD/w_of_LD) that encodes
# Scenario 1 (full weight), Scenario 2 (partial weight), or Scenario 3
# (zero weight / Raman-silent), and forms D = C @ S^T per experiment.
def syn_Raman_no_noise(num_time_steps, Ki, id_of_LD, w_of_LD, RSeed_STchoice):
    
    id_of_LD = np.array(id_of_LD)
    w_of_LD = np.array(w_of_LD)
    concentration_dict = syn_GT_no_noise(num_time_steps, Ki)
    keys = list(concentration_dict.keys())
    
    final_dict = {}
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Randomly Choose Raman Spectra from the BR library
    
    # Number of pure spectra needed = number of species (3).
    num_choose = len(concentration_dict[keys[0]][-1])
    
    filename_spectra = "new_biological_relevant_spectra.pkl"
    with open(filename_spectra, "rb") as file:
        spectra_pure_profile = pickle.load(file)
    keys_spc = list(spectra_pure_profile.keys())

    # Seeded random draw of `num_choose` spectra from the library, subsampling
    # every 12th wavenumber point (::12) to thin the resolution.
    random.seed(RSeed_STchoice)
    select_spectra = random.sample(range(len(keys_spc)), num_choose)
    
    St = []
    wv_meas = spectra_pure_profile[keys_spc[0]][0][::12]
    
    for i in range(num_choose):
        St.append(spectra_pure_profile[keys_spc[select_spectra[i]]][1][::12])
        assert(np.all(wv_meas == spectra_pure_profile[keys_spc[select_spectra[i]]][0][::12]))
    
    St_full = np.array(St)
    
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Randomly Choose Raman Spectra from the BR library
    
    # For each experiment: scale the "low-detectability" species' spectra by
    # w_of_LD (0 for fully silent, <1 for weakly absorbing, 1 for fully
    # absorbing), then form D = C @ S^T.
    for i in range(len(keys)):
        
        t = concentration_dict[keys[i]][0]
        
        y_full = concentration_dict[keys[i]][1]
        
        St = []
        y = []
        for k in range(len(y_full)):
            y.append(y_full[k])
            if k not in id_of_LD:
                St.append(St_full[k])
            elif k in id_of_LD:
                St.append(w_of_LD[np.argmax(id_of_LD == k)]*St_full[k])
                
        y = np.array(y).T
        St = np.array(St)
        D = np.matmul(y, St)
        
        final_dict[keys[i]] = [t, wv_meas, D]
    
    return final_dict

In [5]:
# Adds multiplicative Gaussian noise to the clean synthetic Raman data:
# D_final = D_pure * (1 + noise * per_err/100), with an independent noise
# draw (seeded by RSeed_D_err + experiment index) per experiment.
def syn_Raman_noise(num_time_steps, per_err, Ki, id_of_LD, w_of_LD, RSeed_ST_choice, RSeed_D_err):
    
    Raman_dict = syn_Raman_no_noise(num_time_steps, Ki, id_of_LD, w_of_LD, RSeed_ST_choice)
    keys = list(Raman_dict.keys())
    
    full_dict = {}
    
    for i in range(len(keys)):
        
        t = Raman_dict[keys[i]][0]
        wv = Raman_dict[keys[i]][1]
        D_pure = Raman_dict[keys[i]][2]
        
        D_err = np.random.default_rng(int(RSeed_D_err + i)).standard_normal(D_pure.shape)
        D_final = D_pure * (1 + D_err*(per_err/100))
        
        full_dict[keys[i]] = [t, wv, D_final]
    
    return full_dict

In [6]:
# Recomputes just the (wavenumber grid, ground-truth pure spectra) pair for
# a given low-detectability weighting/seed, applying the same species
# weighting/filtering as syn_Raman_no_noise but without needing the full
# concentration/Raman pipeline -- used to save ST_ground alongside each
# training condition file for later spectral-recovery scoring. Note: a
# species with w_of_LD == 0 is dropped entirely from ST here (matching
# Scenario 3, where that species has no ground-truth spectrum to recover).
def choosen_pure_spectra(id_of_LD, w_of_LD, RSeed_STchoice):
    
    ICs = np.array([[0.5, 0, 0],
                    [0.2, 0, 0]])
    num_choose = len(ICs[0])
    
    filename_spectra = "new_biological_relevant_spectra.pkl"
    with open(filename_spectra, "rb") as file:
        spectra_pure_profile = pickle.load(file)
    keys_spc = list(spectra_pure_profile.keys())
    
    random.seed(RSeed_STchoice)
    select_spectra = random.sample(range(len(keys_spc)), num_choose)
    
    St = []
    wv_meas = spectra_pure_profile[keys_spc[0]][0][::12]
    
    for i in range(num_choose):
        St.append(spectra_pure_profile[keys_spc[select_spectra[i]]][1][::12])
        assert(np.all(wv_meas == spectra_pure_profile[keys_spc[select_spectra[i]]][0][::12]))
    
    St_full = np.array(St)
    
    St = []
    
    for k in range(num_choose):
        if k not in id_of_LD:
            St.append(St_full[k])
        elif k in id_of_LD:
            if w_of_LD[np.argmax(id_of_LD == k)] > 0:
                St.append(w_of_LD[np.argmax(id_of_LD == k)]*St_full[k])
    ST = np.array(St)
    
    return [wv_meas, ST]

# Scenario 1 and 2

In [7]:
# Sweep definition for Scenario 1 (species B fully Raman-active,
# scenario_list value 1) and Scenario 2 (B weakly Raman-active, value
# 0.5): two spectral-overlap regimes (rs_ST_dict keys 0.2/0.6, each with 3
# seeds), 5 noise levels, and 3 inhibition strengths (Ki1 = Ki2).
rs_ST_dict = {0.2 : [28, 20, 736],
              0.6 : [265, 765, 559]}
ky_ST = list(rs_ST_dict.keys())

rs_D_list = [5, 15, 25, 35, 45]

Ki1_list = [0.05, 0.5, 5]

scenario_list = [1, 0.5]

In [8]:
# Cartesian product of every (spectral overlap, spectra seed, noise level,
# inhibition strength, low-detectability weight) combination defined above
# -- one entry per training condition file to be generated.
Test_scenarios = []

for i in range(len(ky_ST)):
    for j in range(len(rs_ST_dict[ky_ST[i]])):
        for k in range(len(rs_D_list)):
            for l in range(len(Ki1_list)):
                for m in range(len(scenario_list)):
                    
                    Ki = tuple([[Ki1_list[l], Ki1_list[l]]])
                    wLD = scenario_list[m]
                    rs_ST = rs_ST_dict[ky_ST[i]][j]
                    rs_D = rs_D_list[k]
                    spc_ovp = ky_ST[i]

                    temp = [Ki, wLD, rs_ST, rs_D, spc_ovp]
                    Test_scenarios.append(temp)

In [9]:
# Sanity-check: total number of Scenario 1 + 2 training conditions.
len(Test_scenarios)

180

In [10]:
# Generates and saves one training condition file per entry in
# Test_scenarios: simulates ground truth, projects to noisy synthetic
# Raman data, computes the matching ground-truth pure spectra, and pickles
# {t_fin, wv_fin, D_full, C0, ST_ground} to "Train Conditions/Scenario
# <1 or 2>/TC_<Ki1>_<Ki2>_<seed>_<overlap>_<noise>.pkl" -- Scenario 1 vs. 2
# is decided purely by whether the low-detectability weight (wLD) is 1
# (full) or 0.5 (partial).
for i in range(len(Test_scenarios)):
    ns = 10
    ne = 5
    
    wLD = Test_scenarios[i][1]
    Ki = Test_scenarios[i][0]
    rs_ST = Test_scenarios[i][2]
    rs_D = Test_scenarios[i][3]
    spc_ovp = Test_scenarios[i][4]
    
    id_of_LD = [1]
    w_of_LD = [wLD]
    
    concentration_dict = syn_GT_no_noise(ns, Ki)
    syn_Raman_dict = syn_Raman_noise(ns, ne, Ki, id_of_LD, w_of_LD, rs_ST, rs_D)
    
    keys = list(syn_Raman_dict.keys())
    
    D_full = []

    for i in range(len(keys)):
        D_full.append(syn_Raman_dict[keys[i]][-1])

    D_full = np.vstack(D_full)
    t_fin = syn_Raman_dict[keys[0]][0]
    wv_fin = syn_Raman_dict[keys[0]][1]
    ICs = np.array([[0.5, 0, 0],[0.2, 0, 0]])
    ST_ground = choosen_pure_spectra(id_of_LD, w_of_LD, rs_ST)[1]
    
    temp_dict = {}
    
    temp_dict["t_fin"] = t_fin
    temp_dict["wv_fin"] = wv_fin
    temp_dict["D_full"] = D_full
    temp_dict["C0"] = ICs
    temp_dict["ST_ground"] = ST_ground
    
    if wLD == 1:
        scn = 1
    elif wLD == 0.5:
        scn = 2
    
    path_name = "Train Conditions/"+"Scenario "+str(scn) + "/"
    file_name = "TC_"+str(Ki[0][0])+"_"+str(Ki[0][1])+"_"+str(rs_ST)+"_"+str(spc_ovp)+"_"+str(rs_D)+".pkl"
    final_name = path_name + file_name

    with open(final_name, "wb") as f:
        pickle.dump(temp_dict, f)


# Scenario 3

In [11]:
# Sweep definition for Scenario 3 (species B fully Raman-silent,
# scenario_list = [0]): same spectral-overlap/noise/inhibition grid as
# Scenario 1 & 2, but with fresh spectra-selection seeds and only the
# w_of_LD = 0 (fully silent) weighting.
rs_ST_dict = {0.2 : [984, 44, 872],
              0.6 : [548, 223, 671]}

ky_ST = list(rs_ST_dict.keys())

rs_D_list = [5, 15, 25, 35, 45]

Ki1_list = [0.05, 0.5, 5]

scenario_list = [0]

In [12]:
# Cartesian product of every Scenario 3 condition combination.
Test_scenarios = []

for i in range(len(ky_ST)):
    for j in range(len(rs_ST_dict[ky_ST[i]])):
        for k in range(len(rs_D_list)):
            for l in range(len(Ki1_list)):
                for m in range(len(scenario_list)):
                    
                    Ki = tuple([[Ki1_list[l], Ki1_list[l]]])
                    wLD = scenario_list[m]
                    rs_ST = rs_ST_dict[ky_ST[i]][j]
                    rs_D = rs_D_list[k]
                    spc_ovp = ky_ST[i]

                    temp = [Ki, wLD, rs_ST, rs_D, spc_ovp]
                    Test_scenarios.append(temp)

In [13]:
# Generates and saves one training condition file per entry in
# Test_scenarios: same pipeline as the Scenario 1/2 loop above, but always
# writes to "Train Conditions/Scenario 3/..." since wLD == 0 for every
# entry here.
for i in range(len(Test_scenarios)):
    ns = 10
    ne = 5
    
    wLD = Test_scenarios[i][1]
    Ki = Test_scenarios[i][0]
    rs_ST = Test_scenarios[i][2]
    rs_D = Test_scenarios[i][3]
    spc_ovp = Test_scenarios[i][4]
    
    id_of_LD = [1]
    w_of_LD = [wLD]
    
    concentration_dict = syn_GT_no_noise(ns, Ki)
    syn_Raman_dict = syn_Raman_noise(ns, ne, Ki, id_of_LD, w_of_LD, rs_ST, rs_D)
    
    keys = list(syn_Raman_dict.keys())
    
    D_full = []

    for i in range(len(keys)):
        D_full.append(syn_Raman_dict[keys[i]][-1])

    D_full = np.vstack(D_full)
    t_fin = syn_Raman_dict[keys[0]][0]
    wv_fin = syn_Raman_dict[keys[0]][1]
    ICs = np.array([[0.5, 0, 0],[0.2, 0, 0]])
    ST_ground = choosen_pure_spectra(id_of_LD, w_of_LD, rs_ST)[1]
    
    temp_dict = {}
    
    temp_dict["t_fin"] = t_fin
    temp_dict["wv_fin"] = wv_fin
    temp_dict["D_full"] = D_full
    temp_dict["C0"] = ICs
    temp_dict["ST_ground"] = ST_ground
    
    scn = 3
    
    path_name = "Train Conditions/"+"Scenario "+str(scn) + "/"
    file_name = "TC_"+str(Ki[0][0])+"_"+str(Ki[0][1])+"_"+str(rs_ST)+"_"+str(spc_ovp)+"_"+str(rs_D)+".pkl"
    final_name = path_name + file_name

    with open(final_name, "wb") as f:
        pickle.dump(temp_dict, f)
